In [ ]:
from pyspark.sql.functions import col, min, max
from pyspark.sql.types import DateType, DecimalType, IntegerType, BooleanType

CATALOG = "czech_fintech"
SILVER  = "silver"

passed = []
failed = []

def check(condition, msg):
    if condition:
        passed.append(msg)
    else:
        failed.append(msg)

def schema_of(df):
    return {f.name: type(f.dataType) for f in df.schema.fields}

In [ ]:
print("=" * 80)
print("SILVER LAYER QUALITY CHECKS")
print("=" * 80)

# ── TRANSACTIONS ─────────────────────────────────────────────────────────────
print("\n1️⃣  TRANSACTIONS")
trans        = spark.table(f"{CATALOG}.{SILVER}.transactions")
trans_count  = trans.count()
distinct_ids = trans.select("transaction_id").distinct().count()
dup_count    = trans_count - distinct_ids
nulls_client = trans.filter(col("client_id").isNull()).count()
pct_client   = round((trans_count - nulls_client) / trans_count * 100, 1)
date_min, date_max = trans.select(min("date"), max("date")).first()
s = schema_of(trans)

check(trans_count > 0,              f"transactions non vuota ({trans_count} rows)")
check(dup_count == 0,               f"transaction_id unico — {dup_count} duplicati nel sorgente")
check(s["date"]    == DateType,     "date → DateType")
check(s["amount"]  == DecimalType,  "amount → DecimalType")
check(s["balance"] == DecimalType,  "balance → DecimalType")
check(pct_client >= 95,             f"client_id copertura {pct_client}% (soglia 95%)")
check(str(date_min)[:4] == "1993",  f"date_min {date_min}")
check(str(date_max)[:4] <= "1999",  f"date_max {date_max}")
check("year"  in trans.columns,     "colonna year presente")
check("month" in trans.columns,     "colonna month presente")

trans.select(min("date").alias("date_min"), max("date").alias("date_max"),
             min("amount").alias("amt_min"), max("amount").alias("amt_max")).show()

# ── ACCOUNT ──────────────────────────────────────────────────────────────────
print("\n2️⃣  ACCOUNT")
account       = spark.table(f"{CATALOG}.{SILVER}.account")
account_count = account.count()
dupes_scd     = (account.filter(col("is_current") == True)
                 .groupBy("account_id").count()
                 .filter(col("count") > 1).count())
s = schema_of(account)

check(account_count > 0,            f"account non vuota ({account_count} rows)")
check(s["date"]       == DateType,  "date → DateType")
check(s["is_current"] == BooleanType, "is_current → BooleanType")
check("valid_from" in account.columns, "colonna SCD valid_from presente")
check("valid_to"   in account.columns, "colonna SCD valid_to presente")
check(dupes_scd == 0,               f"nessun duplicato is_current=true ({dupes_scd} violazioni)")
check(account.filter(col("account_id").isNull()).count() == 0, "account_id senza NULL")

account.groupBy("frequency").count().orderBy("frequency").show()

# ── CLIENT ───────────────────────────────────────────────────────────────────
print("\n3️⃣  CLIENT")
client      = spark.table(f"{CATALOG}.{SILVER}.client")
bad_gender  = client.filter(~col("gender").isin("M", "F")).count()
bd_min, bd_max = client.select(min("birth_date"), max("birth_date")).first()
gd          = {r["gender"]: r["count"] for r in client.groupBy("gender").count().collect()}
s = schema_of(client)

check(client.count() > 0,           f"client non vuota ({client.count()} rows)")
check(s["birth_date"] == DateType,  "birth_date → DateType")
check(bad_gender == 0,              "gender contiene solo M/F")
check(gd.get("M", 0) > 0,          f"clienti M: {gd.get('M',0)}")
check(gd.get("F", 0) > 0,          f"clienti F: {gd.get('F',0)}")
check(str(bd_min)[:4] >= "1910",    f"birth_date_min {bd_min}")
check(str(bd_max)[:4] <= "1990",    f"birth_date_max {bd_max}")

client.groupBy("gender").count().show()

# ── LOAN ─────────────────────────────────────────────────────────────────────
print("\n4️⃣  LOAN")
loan           = spark.table(f"{CATALOG}.{SILVER}.loan")
bad_status     = loan.filter(~col("status").isin("A", "B", "C", "D")).count()
s = schema_of(loan)

check(loan.count() > 0,             f"loan non vuota ({loan.count()} rows)")
check(s["date"]     == DateType,    "date → DateType")
check(s["amount"]   == DecimalType, "amount → DecimalType")
check(s["duration"] == IntegerType, "duration → IntegerType")
check(s["payments"] == DecimalType, "payments → DecimalType")
check(bad_status == 0,              "loan status solo A/B/C/D")

loan.groupBy("status").count().orderBy("status").show()

# ── ORDER ────────────────────────────────────────────────────────────────────
print("\n5️⃣  ORDER")
order = spark.table(f"{CATALOG}.{SILVER}.order")
s = schema_of(order)

check(order.count() > 0,            f"order non vuota ({order.count()} rows)")
check(s["amount"] == DecimalType,   "amount → DecimalType")
check(order.filter(col("amount").isNull()).count() == 0, "amount senza NULL")

# ── CARD ─────────────────────────────────────────────────────────────────────
print("\n6️⃣  CARD")
card = spark.table(f"{CATALOG}.{SILVER}.card")
s = schema_of(card)

check(card.count() > 0,             f"card non vuota ({card.count()} rows)")
check(s["issued"] == DateType,      "issued → DateType")

card.groupBy("type").count().show()

# ── DISP ─────────────────────────────────────────────────────────────────────
print("\n7️⃣  DISP")
disp     = spark.table(f"{CATALOG}.{SILVER}.disp")
bad_disp = disp.filter(~col("type").isin("OWNER", "DISPONENT")).count()

check(disp.count() > 0,             f"disp non vuota ({disp.count()} rows)")
check(bad_disp == 0,                "disp type solo OWNER/DISPONENT")

# ── DISTRICT ─────────────────────────────────────────────────────────────────
print("\n8️⃣  DISTRICT")
district = spark.table(f"{CATALOG}.{SILVER}.district")

check(district.count() > 0,         f"district non vuota ({district.count()} rows)")
check(district.filter(col("A1").isNull()).count() == 0, "A1 senza NULL")

district.select("A1", "A2", "A3").limit(5).show()

# ── JOIN COVERAGE ─────────────────────────────────────────────────────────────
print("\n9️⃣  JOIN COVERAGE")
matched_account  = trans.join(account.filter(col("is_current")).select("account_id"), "account_id", "inner").count()
pct_account      = round(matched_account / trans_count * 100, 1)
account_total    = account.filter(col("is_current")).count()
matched_district = (account.filter(col("is_current"))
                    .join(district.select(col("A1").alias("district_id")), "district_id", "inner")
                    .count())
pct_district     = round(matched_district / account_total * 100, 1)

check(pct_account  == 100.0, f"transactions → account {pct_account}%")
check(pct_client   >= 95,    f"transactions → client {pct_client}%")
check(pct_district == 100.0, f"account → district {pct_district}%")

# ── RIEPILOGO ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"✅ PASSED: {len(passed)}   ❌ FAILED: {len(failed)}")
print("=" * 80)
for m in passed: print(f"  ✅ {m}")
for m in failed: print(f"  ❌ {m}")
if failed:
    raise AssertionError(f"{len(failed)} check falliti")

In [ ]:
# ============================================================================
# 9. JOIN COVERAGE — transactions → account → client → district
# ============================================================================
print("\n9️⃣  JOIN COVERAGE")

trans_total = trans.count()

# transactions → account
matched_account = trans.join(account.filter(col("is_current")).select("account_id"), "account_id", "inner").count()
pct_account = round(matched_account / trans_total * 100, 1)
assert_check(pct_account == 100.0, f"100% transactions matchano un account ({pct_account}%)")

# transactions → client (via client_id già joinato in silver)
matched_client = trans.filter(col("client_id").isNotNull()).count()
pct_client = round(matched_client / trans_total * 100, 1)
assert_check(pct_client >= 95, f"client_id copertura ≥ 95% ({pct_client}%)")

# account → district
account_total = account.filter(col("is_current")).count()
matched_district = (account.filter(col("is_current"))
                    .join(district.select(col("A1").alias("district_id")), "district_id", "inner")
                    .count())
pct_district = round(matched_district / account_total * 100, 1)
assert_check(pct_district == 100.0, f"100% account matchano un district ({pct_district}%)")

print("\n" + "=" * 80)
print("✅ ALL SILVER QUALITY CHECKS PASSED")
print("=" * 80)